In [0]:
from pyspark.sql import SparkSession

dbutils.widgets.dropdown("environment", "test",["test","prod"])
env =dbutils.widgets.get("environment")

bronze_path= f"s3://fraud-transection-detection/{env}/raw-landing/"
schema_path = f"s3://fraud-transection-detection/{env}/schema"
checkpointloc_raw = f"s3://fraud-transection-detection/{env}/checkpoint/read_raw"
checkpoint_write=f"s3://fraud-transection-detection/{env}/checkpoint/write_silver"
db_name = f"fraud_{env}"


#spark.sql(f"DROP SCHEMA IF EXISTS {db_name} CASCADE") # Clear the old 'default' metadata
spark.sql(f"CREATE SCHEMA if not exists {db_name} Managed LOCATION 's3://fraud-transection-detection/{env}/bronze/'")

db=spark.readStream.format("cloudFiles")\
       .option("cloudFiles.format","json")\
       .option("cloudFiles.schemaHints", "tr_date date")\
       .option("cloudFiles.rescuedDataColumn","rescued_data")\
       .option("cloudFiles.schemaLocation",schema_path)\
       .load(bronze_path)

#display(db,checkpointLocation=checkpointloc)
db.writeStream.format("delta")\
        .option("checkpointLocation",checkpoint_write)\
        .option("mergeSchema",True)\
        .outputMode("append")\
        .trigger(availableNow=True)\
        .toTable(f"{db_name}.bronze")


In [0]:
%sql
SELECT * FROM fraud_test.bronze